# Cottonil | Data Understanding, Quality Audit & Business Analytics

**Project:** Cottonil Enterprise Analytics  
**Notebook purpose:** document the source data, validate its structure and relationships, reconcile business transactions, prepare analysis-ready tables, and reproduce the analytical foundations behind the current Streamlit application.

This notebook is an auditable analysis workflow, not a replacement for the application. It keeps raw source files unchanged and creates derived DataFrames only in memory.

**Workflow**
1. Data inventory and schema review
2. Data quality and key/relationship validation
3. Sales, payments, returns, delivery and supply-chain reconciliation
4. Data preparation and KPI definitions
5. Enterprise EDA
6. Advanced analytics aligned with the six current AI applications
7. Findings, limitations and reproducibility notes

> Run the notebook from the project folder or update `DATA_DIR` in the setup cell. The source CSV files are not modified.

## 1. Project setup and data inventory
Set the data root once. The notebook supports an environment variable, the original local Windows path, and a relative project layout.

In [ ]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
warnings.filterwarnings("ignore", category=FutureWarning)

CANDIDATE_PATHS = [
    Path(os.getenv("COTTONIL_DATA_DIR", "")) if os.getenv("COTTONIL_DATA_DIR") else None,
    Path(r"D:\HVIA\project_two\Cottonil_project\COTTONIL\COTTONIL_Data"),
    Path.cwd() / "COTTONIL_Data",
    Path.cwd() / "COTTONIL" / "COTTONIL_Data",
    Path.cwd() / "data" / "COTTONIL_Data",
]
DATA_DIR = next((p for p in CANDIDATE_PATHS if p and p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate COTTONIL_Data. Set COTTONIL_DATA_DIR to the folder "
        "that contains 01_CATALOG and 02_TABLES."
    )

TABLES_DIR = DATA_DIR / "02_TABLES"
CATALOG_DIR = DATA_DIR / "01_CATALOG"
if not TABLES_DIR.exists():
    raise FileNotFoundError(f"Expected table folder was not found: {TABLES_DIR}")

print("Data root:", DATA_DIR)
print("Tables folder:", TABLES_DIR)
print("Catalog folder exists:", CATALOG_DIR.exists())

In [ ]:
csv_files = sorted(DATA_DIR.rglob("*.csv"))
inventory = pd.DataFrame([
    {
        "file_name": p.name,
        "relative_path": str(p.relative_to(DATA_DIR)),
        "size_MB": round(p.stat().st_size / (1024 ** 2), 3),
    }
    for p in csv_files
])
display(inventory)
print(f"CSV files discovered: {len(inventory)}")

## 2. Data dictionary and schema audit
Compare the documented fields with the actual CSV headers. A mismatch is reported for investigation; the notebook does not silently alter source schemas.

In [ ]:
dictionary_path = TABLES_DIR / "data_dictionary.csv"
dictionary = pd.read_csv(dictionary_path, low_memory=False)
required_dictionary_cols = {"table_name", "field_name", "description"}
missing_dictionary_cols = required_dictionary_cols - set(dictionary.columns)
if missing_dictionary_cols:
    raise ValueError(f"Unexpected data dictionary schema. Missing: {missing_dictionary_cols}")

fields_per_table = (dictionary.groupby("table_name")
                    .agg(documented_fields=("field_name", "count"),
                         unique_fields=("field_name", "nunique"))
                    .sort_values("documented_fields", ascending=False))
print("Tables in dictionary:", dictionary["table_name"].nunique())
print("Field definitions:", len(dictionary))
display(fields_per_table)
print("Duplicate dictionary rows:", dictionary.duplicated().sum())
display(dictionary[dictionary.duplicated(["table_name", "field_name"], keep=False)]
        .sort_values(["table_name", "field_name"]))

In [ ]:
schema_comparison = []
for table_name, group in dictionary.groupby("table_name"):
    path = TABLES_DIR / table_name
    if not path.exists():
        schema_comparison.append({"table": table_name, "status": "FILE NOT FOUND",
                                  "documented_columns": group["field_name"].nunique(),
                                  "actual_columns": np.nan,
                                  "missing_from_file": None, "not_in_dictionary": None})
        continue
    actual = set(pd.read_csv(path, nrows=0).columns)
    documented = set(group["field_name"].dropna())
    schema_comparison.append({
        "table": table_name,
        "status": "OK" if actual == documented else "MISMATCH",
        "documented_columns": len(documented),
        "actual_columns": len(actual),
        "missing_from_file": sorted(documented - actual),
        "not_in_dictionary": sorted(actual - documented),
    })
schema_comparison = pd.DataFrame(schema_comparison).sort_values(["status", "table"])
display(schema_comparison)

## 3. Table profiling and data quality
The following reusable profiler checks row counts, data types, missing values, blank strings, duplicate rows and duplicate candidate keys. It is a profiling layer; business-specific validity rules are handled separately.

In [ ]:
def load_table(filename, **kwargs):
    path = TABLES_DIR / filename
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path, low_memory=False, **kwargs)

def profile_table(df, table_name, key=None):
    row = {
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "blank_string_cells": int(sum(
            df[c].astype("string").str.strip().eq("").fillna(False).sum()
            for c in df.select_dtypes(include=["object", "string"]).columns
        )),
    }
    if key and key in df:
        row["key"] = key
        row["missing_key"] = int(df[key].isna().sum() + df[key].astype("string").str.strip().eq("").sum())
        row["duplicate_key_rows"] = int(df[key].duplicated(keep=False).sum())
    return row

profile_rows = []
for path in sorted(TABLES_DIR.glob("*.csv")):
    if path.name == "data_dictionary.csv":
        continue
    frame = load_table(path.name)
    profile_rows.append(profile_table(frame, path.name))
quality_profile = pd.DataFrame(profile_rows).sort_values("rows", ascending=False)
display(quality_profile)

### 3.1 Primary-key and foreign-key checks
The checks below separate missing references from nonblank references that do not exist in the parent table. Missing customer IDs are an observed business-data condition, not a reason to delete sales.

In [ ]:
PRIMARY_KEYS = {
    "branches.csv": "branch_id",
    "customers.csv": "customer_id",
    "sales_orders.csv": "sales_order_id",
    "sales_order_items.csv": "sales_line_id",
    "sales_payments.csv": "payment_id",
    "sales_returns.csv": "return_id",
    "purchase_orders.csv": "po_id",
    "purchase_order_items.csv": "po_line_id",
    "employees.csv": "employee_id",
    "payroll.csv": "payroll_id",
    "delivery_shipments.csv": "shipment_id",
    "inventory_transfers.csv": "transfer_id",
    "inventory_transfer_items.csv": "transfer_line_id",
}

pk_results = []
for filename, key in PRIMARY_KEYS.items():
    path = TABLES_DIR / filename
    if not path.exists():
        continue
    s = pd.read_csv(path, usecols=[key], dtype="string", low_memory=False)[key].str.strip()
    pk_results.append({
        "table": filename, "key": key, "rows": len(s),
        "missing_or_blank": int((s.isna() | s.eq("")).sum()),
        "duplicate_key_rows": int(s.duplicated(keep=False).sum()),
        "unique_nonblank_keys": int(s[s.notna() & s.ne("")].nunique()),
    })
pk_audit = pd.DataFrame(pk_results)
display(pk_audit)

In [ ]:
def key_series(filename, column):
    return load_table(filename, usecols=[column], dtype="string")[column].str.strip()

def relationship_audit(child_file, child_col, parent_file, parent_col):
    child = key_series(child_file, child_col)
    parent = key_series(parent_file, parent_col)
    missing = child.isna() | child.eq("")
    unmatched = (~missing) & (~child.isin(set(parent.dropna())))
    return {
        "relationship": f"{child_file}.{child_col} -> {parent_file}.{parent_col}",
        "rows_checked": len(child),
        "missing_or_blank": int(missing.sum()),
        "unmatched_nonblank": int(unmatched.sum()),
        "unmatched_examples": child[unmatched].drop_duplicates().head(5).tolist(),
    }

relationships = [
    ("sales_orders.csv", "branch_id", "branches.csv", "branch_id"),
    ("sales_orders.csv", "customer_id", "customers.csv", "customer_id"),
    ("sales_order_items.csv", "sales_order_id", "sales_orders.csv", "sales_order_id"),
    ("sales_payments.csv", "sales_order_id", "sales_orders.csv", "sales_order_id"),
    ("sales_returns.csv", "sales_order_id", "sales_orders.csv", "sales_order_id"),
    ("sales_returns.csv", "sales_line_id", "sales_order_items.csv", "sales_line_id"),
    ("purchase_orders.csv", "supplier_id", "suppliers.csv", "supplier_id"),
    ("purchase_orders.csv", "receiving_warehouse_id", "warehouses.csv", "warehouse_id"),
]
relationship_audit_df = pd.DataFrame([
    relationship_audit(*r) for r in relationships
    if all((TABLES_DIR / f).exists() for f in [r[0], r[2]])
])
display(relationship_audit_df)

## 4. Transaction validation and reconciliation

### 4.1 Sales orders and sales order items
Validate order keys, date coverage, financial fields and the reconciliation between order-header totals and the sum of line-level amounts.

In [ ]:
orders = load_table("sales_orders.csv", dtype={"sales_order_id": "string", "customer_id": "string"})
sales_items = load_table("sales_order_items.csv", dtype={"sales_order_id": "string", "sales_line_id": "string", "sku_id": "string"})

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
sales_items["order_date"] = pd.to_datetime(sales_items["order_date"], errors="coerce")

print("Orders:", len(orders), "| unique order IDs:", orders["sales_order_id"].nunique())
print("Order date range:", orders["order_date"].min(), "to", orders["order_date"].max())
print("Invalid order dates:", int(orders["order_date"].isna().sum()))
print("Sales lines:", len(sales_items), "| unique line IDs:", sales_items["sales_line_id"].nunique())
print("Invalid line dates:", int(sales_items["order_date"].isna().sum()))

money_order_cols = [c for c in ["gross_EGP", "discount_EGP", "net_before_tax_EGP", "tax_EGP", "paid_total_EGP"] if c in orders]
money_line_cols = [c for c in ["gross_line_EGP", "discount_line_EGP", "net_line_EGP", "unit_cost_EGP", "margin_EGP"] if c in sales_items]
display(orders[money_order_cols].describe().T)
display(sales_items[money_line_cols].describe().T)
print("Non-positive quantities:", int((pd.to_numeric(sales_items["quantity"], errors="coerce") <= 0).sum()))

In [ ]:
line_totals = (sales_items.groupby("sales_order_id", as_index=False)
               .agg(items_gross=("gross_line_EGP", "sum"),
                    items_discount=("discount_line_EGP", "sum"),
                    items_net=("net_line_EGP", "sum")))
order_line_check = orders[["sales_order_id", "gross_EGP", "discount_EGP", "net_before_tax_EGP"]].merge(
    line_totals, on="sales_order_id", how="left", validate="one_to_one", indicator=True
)
for header, detail in [("gross_EGP", "items_gross"),
                       ("discount_EGP", "items_discount"),
                       ("net_before_tax_EGP", "items_net")]:
    order_line_check[f"{header}_difference"] = (
        pd.to_numeric(order_line_check[header], errors="coerce")
        - pd.to_numeric(order_line_check[detail], errors="coerce")
    )
reconciliation_summary = pd.DataFrame([
    {"metric": col, "mismatch_rows": int((~np.isclose(order_line_check[col].fillna(0), 0, atol=0.01)).sum()),
     "max_abs_difference_EGP": float(order_line_check[col].abs().max())}
    for col in [c for c in order_line_check.columns if c.endswith("_difference")]
])
print("Order-to-lines merge status:")
display(order_line_check["_merge"].value_counts().rename_axis("merge_status").to_frame("orders"))
display(reconciliation_summary)

### 4.2 Customer ID coverage
Customer-level analysis must use identified customers only. Orders without a customer ID remain included in company-wide sales and order totals.

In [ ]:
customer_id_clean = orders["customer_id"].astype("string").str.strip()
missing_customer = customer_id_clean.isna() | customer_id_clean.eq("")
orders["customer_id_clean"] = customer_id_clean.mask(missing_customer)
print("Total orders:", len(orders))
print("Orders with missing customer ID:", int(missing_customer.sum()))
print("Missing customer ID rate (%):", round(missing_customer.mean() * 100, 2))
if "sales_channel" in orders:
    display(pd.crosstab(orders["sales_channel"].fillna("Missing"),
                        missing_customer.map({True: "Missing ID", False: "Identified"}),
                        margins=True))

### 4.3 Payments and returns

In [ ]:
payments = load_table("sales_payments.csv", dtype={"sales_order_id": "string"})
payments["payment_date"] = pd.to_datetime(payments["payment_date"], errors="coerce")
print("Payments:", len(payments), "| unique payment IDs:", payments["payment_id"].nunique())
print("Payment date range:", payments["payment_date"].min(), "to", payments["payment_date"].max())
display(payments["status"].value_counts(dropna=False).rename_axis("payment_status").to_frame("records"))
display(payments.groupby("payment_method", dropna=False)["amount_EGP"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False))

returns = load_table("sales_returns.csv", dtype={"sales_order_id": "string", "sales_line_id": "string"})
returns["return_date"] = pd.to_datetime(returns["return_date"], errors="coerce")
print("Returns:", len(returns), "| unique return IDs:", returns["return_id"].nunique())
print("Return date range:", returns["return_date"].min(), "to", returns["return_date"].max())
display(returns.groupby("reason", dropna=False).agg(
    return_records=("return_id", "count"),
    returned_units=("return_quantity", "sum"),
    refund_EGP=("refund_amount_EGP", "sum")
).sort_values("refund_EGP", ascending=False))

### 4.4 Delivery performance
Calculate elapsed shipping-to-delivery time only when both dates are valid. No late-delivery/SLA breach conclusion is made because a promised delivery date or formal SLA definition is not assumed.

In [ ]:
delivery = load_table("delivery_shipments.csv", dtype={"sales_order_id": "string"})
for col in ["ship_date", "delivery_date"]:
    if col in delivery:
        delivery[col] = pd.to_datetime(delivery[col], errors="coerce")
if {"ship_date", "delivery_date"}.issubset(delivery.columns):
    delivery["transit_days"] = (delivery["delivery_date"] - delivery["ship_date"]).dt.total_seconds() / 86400
    print("Negative transit intervals:", int((delivery["transit_days"] < 0).sum()))
    display(delivery["transit_days"].describe().to_frame("transit_days"))
if "fulfillment_status" in delivery:
    display(delivery["fulfillment_status"].value_counts(dropna=False).to_frame("shipments"))

### 4.5 Procurement, suppliers and inventory movement
Inspect purchase-order structure, supplier/warehouse references and transfer direction. The available fields do not necessarily represent actual received quantities.

In [ ]:
po = load_table("purchase_orders.csv", dtype={"po_id": "string", "supplier_id": "string"})
po_items = load_table("purchase_order_items.csv", dtype={"po_id": "string", "sku_id": "string"})
print("Purchase orders:", len(po), "| unique IDs:", po["po_id"].nunique())
print("Purchase order lines:", len(po_items), "| unique line IDs:", po_items["po_line_id"].nunique())
display(po["po_status"].value_counts(dropna=False).to_frame("purchase_orders"))
display(po_items.groupby("po_id").size().describe().to_frame("items_per_PO"))
if "delivery_date" in po_items:
    po_items["delivery_date"] = pd.to_datetime(po_items["delivery_date"], errors="coerce")
    print("PO line delivery date range:", po_items["delivery_date"].min(), "to", po_items["delivery_date"].max())

stock = load_table("monthly_stock.csv")
transfers = load_table("inventory_transfers.csv")
transfer_items = load_table("inventory_transfer_items.csv")
print("Monthly stock:", stock.shape, "| Transfers:", transfers.shape, "| Transfer lines:", transfer_items.shape)
display(stock.head(3))
display(transfers.head(3))

## 5. Data preparation layer
Create analysis-ready copies with normalized identifiers and parsed dates. Keep the original DataFrames and source files intact. Missing customer identifiers are retained as missing, not replaced with fabricated IDs.

In [ ]:
orders_clean = orders.copy()
items_clean = sales_items.copy()
returns_clean = returns.copy()
payments_clean = payments.copy()

for df in [orders_clean, items_clean, returns_clean, payments_clean]:
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype("string").str.strip()

# Numeric coercion is explicit and limited to known measures.
numeric_columns = {
    "orders": ["gross_EGP", "discount_EGP", "net_before_tax_EGP", "tax_EGP", "paid_total_EGP"],
    "items": ["quantity", "unit_price_EGP", "gross_line_EGP", "discount_line_EGP",
              "net_line_EGP", "unit_cost_EGP", "margin_EGP"],
    "returns": ["return_quantity", "refund_amount_EGP"],
    "payments": ["amount_EGP"],
}
for name, frame in [("orders", orders_clean), ("items", items_clean),
                    ("returns", returns_clean), ("payments", payments_clean)]:
    for col in numeric_columns[name]:
        if col in frame:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")

print("Prepared copies created:", {
    "orders": orders_clean.shape, "items": items_clean.shape,
    "returns": returns_clean.shape, "payments": payments_clean.shape
})

## 6. Enterprise exploratory data analysis

### 6.1 Company-level commercial KPIs
Use order headers for order-level financial totals and line items for unit/margin analysis. Avoid summing header totals after joining to multiple line items.

In [ ]:
kpi = {
    "Orders": orders_clean["sales_order_id"].nunique(),
    "Gross sales (EGP)": orders_clean["gross_EGP"].sum(),
    "Discounts (EGP)": orders_clean["discount_EGP"].sum(),
    "Net before tax (EGP)": orders_clean["net_before_tax_EGP"].sum(),
    "Paid total (EGP)": orders_clean["paid_total_EGP"].sum(),
    "Units sold": items_clean["quantity"].sum(),
    "Recorded line margin (EGP)": items_clean["margin_EGP"].sum(),
    "Return records": returns_clean["return_id"].nunique(),
    "Returned units": returns_clean["return_quantity"].sum(),
    "Refunds (EGP)": returns_clean["refund_amount_EGP"].sum(),
}
display(pd.Series(kpi, name="value").to_frame())
if items_clean["quantity"].sum() != 0:
    print("Unit return rate (%):",
          round(returns_clean["return_quantity"].sum() / items_clean["quantity"].sum() * 100, 2))

In [ ]:
orders_monthly = (orders_clean.dropna(subset=["order_date"])
                  .assign(year_month=lambda x: x["order_date"].dt.to_period("M").dt.to_timestamp())
                  .groupby("year_month", as_index=False)
                  .agg(orders=("sales_order_id", "nunique"),
                       gross_sales_EGP=("gross_EGP", "sum"),
                       net_sales_EGP=("net_before_tax_EGP", "sum"),
                       paid_EGP=("paid_total_EGP", "sum")))
display(orders_monthly.tail(18))
ax = orders_monthly.set_index("year_month")["net_sales_EGP"].plot(
    figsize=(13, 4), title="Monthly Net Sales (EGP)", marker="o"
)
ax.set_xlabel("Order month"); ax.set_ylabel("EGP"); plt.tight_layout(); plt.show()

### 6.2 Branch and product/category performance

In [ ]:
if "branch_id" in orders_clean:
    branch_sales = (orders_clean.groupby("branch_id", as_index=False)
                    .agg(orders=("sales_order_id", "nunique"),
                         net_sales_EGP=("net_before_tax_EGP", "sum"),
                         paid_EGP=("paid_total_EGP", "sum"))
                    .sort_values("net_sales_EGP", ascending=False))
    display(branch_sales.head(20))

products = load_table("products_catalog.csv")
skus = load_table("sku_variants.csv")
item_product = items_clean.merge(
    skus[["sku_id", "product_id"]], on="sku_id", how="left", validate="many_to_one"
).merge(
    products[["product_id"] + [c for c in ["product_name_ar", "product_name_en", "category_en", "listed_price_EGP"] if c in products]],
    on="product_id", how="left", validate="many_to_one"
)
category_col = "category_en" if "category_en" in item_product else None
if category_col:
    category_performance = (item_product.groupby(category_col, dropna=False)
                            .agg(units=("quantity", "sum"),
                                 net_sales_EGP=("net_line_EGP", "sum"),
                                 margin_EGP=("margin_EGP", "sum"))
                            .sort_values("net_sales_EGP", ascending=False))
    display(category_performance)

### 6.3 Marketing campaign descriptive review
Campaign-level spend, budget, impressions, clicks and available outcomes are descriptive indicators. Without a reliable order-to-campaign key, do not interpret them as causal attribution.

In [ ]:
campaigns = load_table("marketing_campaigns.csv")
display(campaigns.head())
campaign_numeric = [c for c in campaigns.select_dtypes(include=np.number).columns]
display(campaigns[campaign_numeric].describe().T if campaign_numeric else pd.DataFrame())
if {"spend_EGP", "budget_EGP"}.issubset(campaigns.columns):
    campaigns["budget_utilization_pct"] = np.where(
        campaigns["budget_EGP"] > 0, campaigns["spend_EGP"] / campaigns["budget_EGP"] * 100, np.nan
    )
    display(campaigns[["campaign_id", "budget_EGP", "spend_EGP", "budget_utilization_pct"]]
            .sort_values("spend_EGP", ascending=False).head(15))

## 7. Advanced analytics aligned with the current Cottonil application

### 7.1 Customer Win-Back / RFM segmentation
RFM is computed only for orders with a valid customer ID. The recency reference date is one day after the latest observed order date. Scores are relative quintiles within this dataset, not permanent customer labels.

In [ ]:
rfm_orders = orders_clean.dropna(subset=["customer_id_clean", "order_date"]).copy()
rfm_orders = rfm_orders[rfm_orders["customer_id_clean"].ne("")]
reference_date = rfm_orders["order_date"].max() + pd.Timedelta(days=1)
rfm = (rfm_orders.groupby("customer_id_clean")
       .agg(Recency=("order_date", lambda s: (reference_date - s.max()).days),
            Frequency=("sales_order_id", "nunique"),
            Monetary=("paid_total_EGP", "sum"))
       .reset_index())
def quintile_score(series, ascending=True):
    # rank(method='first') avoids qcut failures when values repeat heavily
    ranks = series.rank(method="first", ascending=ascending)
    return pd.qcut(ranks, 5, labels=[1, 2, 3, 4, 5]).astype(int)

if len(rfm) >= 5:
    rfm["R_score"] = quintile_score(rfm["Recency"], ascending=False)
    rfm["F_score"] = quintile_score(rfm["Frequency"], ascending=True)
    rfm["M_score"] = quintile_score(rfm["Monetary"], ascending=True)
    rfm["RFM_score"] = rfm[["R_score", "F_score", "M_score"]].sum(axis=1)
    rfm["segment"] = np.select(
        [rfm["RFM_score"] >= 13, rfm["RFM_score"] >= 10,
         rfm["RFM_score"] >= 7, rfm["RFM_score"] >= 5],
        ["Champions / high value", "Loyal / promising", "Needs attention", "At risk"],
        default="Low engagement"
    )
    display(rfm.groupby("segment").agg(
        customers=("customer_id_clean", "nunique"),
        avg_recency_days=("Recency", "mean"),
        avg_frequency=("Frequency", "mean"),
        total_monetary_EGP=("Monetary", "sum")
    ).sort_values("customers", ascending=False))
else:
    print("Not enough identified customers to create quintile-based RFM scores.")
print("Identified customers used for RFM:", rfm["customer_id_clean"].nunique())
print("Orders excluded from customer-level RFM due to missing customer ID:",
      int(orders_clean["customer_id_clean"].isna().sum()))

### 7.2 AI Demand & Inventory Intelligence — monthly demand forecast
The workflow mirrors the current app's methodology: monthly unit demand, continuous month index, calendar/lag/rolling features, Random Forest, six most recent one-step-ahead validation months, and a seasonal-naive lag-12 benchmark. Forecasting is run per product category only when at least 30 monthly observations exist. Results should be compared with the app output after execution.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Build category-level monthly demand from line items joined to product categories.
if "category_en" not in item_product.columns:
    raise KeyError("category_en is required for category-level demand forecasting.")
demand = item_product.dropna(subset=["order_date"]).copy()
demand["month"] = demand["order_date"].dt.to_period("M").dt.to_timestamp()
monthly_demand = (demand.groupby(["category_en", "month"])["quantity"]
                  .sum().rename("units").reset_index())

FORECAST_FEATURES = [
    "month_sin", "month_cos", "time_index",
    "lag_1", "lag_2", "lag_3", "lag_6", "lag_12",
    "rolling_mean_3", "rolling_mean_6", "rolling_mean_12",
]

def make_features(history, target_date, time_index):
    h = np.asarray(history, dtype=float)
    return {
        "month_sin": np.sin(2 * np.pi * target_date.month / 12),
        "month_cos": np.cos(2 * np.pi * target_date.month / 12),
        "time_index": float(time_index),
        "lag_1": float(h[-1]), "lag_2": float(h[-2]),
        "lag_3": float(h[-3]), "lag_6": float(h[-6]),
        "lag_12": float(h[-12]),
        "rolling_mean_3": float(np.mean(h[-3:])),
        "rolling_mean_6": float(np.mean(h[-6:])),
        "rolling_mean_12": float(np.mean(h[-12:])),
    }

def forecast_category(series, horizon=3):
    y = pd.to_numeric(series, errors="coerce").astype(float).clip(lower=0).dropna()
    y.index = pd.to_datetime(y.index).to_period("M").to_timestamp()
    y = y.groupby(level=0).sum().sort_index()
    if len(y) < 30:
        return None, {"status": "Skipped: fewer than 30 months", "months": len(y)}
    full_index = pd.date_range(y.index.min(), y.index.max(), freq="MS")
    y = y.reindex(full_index, fill_value=0.0)
    values = y.to_numpy()
    X, target = [], []
    for i in range(12, len(values)):
        X.append(make_features(values[:i], full_index[i], i))
        target.append(values[i])
    X_all = pd.DataFrame(X, columns=FORECAST_FEATURES)
    y_all = np.asarray(target, dtype=float)

    actual, predicted, naive = [], [], []
    for pos in range(max(24, len(values) - 6), len(values)):
        train_end = pos - 12
        if train_end < 12:
            continue
        model = RandomForestRegressor(
            n_estimators=220, max_depth=7, min_samples_leaf=2,
            max_features=0.9, random_state=42, n_jobs=-1
        )
        model.fit(X_all.iloc[:train_end], y_all[:train_end])
        x_test = pd.DataFrame([make_features(values[:pos], full_index[pos], pos)])
        predicted.append(max(0.0, float(model.predict(x_test)[0])))
        actual.append(float(values[pos]))
        naive.append(float(values[pos - 12]))
    if len(actual) < 3:
        return None, {"status": "Skipped: insufficient backtest points", "months": len(y)}
    actual_arr, pred_arr, naive_arr = map(np.asarray, (actual, predicted, naive))
    denom = np.abs(actual_arr).sum()
    metrics = {
        "status": "Evaluated", "months": len(y), "backtest_months": len(actual),
        "RF_MAE": mean_absolute_error(actual_arr, pred_arr),
        "RF_WAPE_pct": np.abs(actual_arr - pred_arr).sum() / denom * 100 if denom else np.nan,
        "Seasonal_Naive_WAPE_pct": np.abs(actual_arr - naive_arr).sum() / denom * 100 if denom else np.nan,
    }
    final_model = RandomForestRegressor(
        n_estimators=220, max_depth=7, min_samples_leaf=2,
        max_features=0.9, random_state=42, n_jobs=-1
    )
    final_model.fit(X_all, y_all)
    future_dates = pd.date_range(full_index[-1] + pd.offsets.MonthBegin(1), periods=horizon, freq="MS")
    history = list(values)
    future = []
    for step, date in enumerate(future_dates):
        pred = max(0.0, float(final_model.predict(
            pd.DataFrame([make_features(history, date, len(values) + step)])
        )[0]))
        future.append(pred)
        history.append(pred)
    return pd.DataFrame({"month": future_dates, "forecast_units": future}), metrics

forecast_outputs, forecast_metrics = [], []
for category, group in monthly_demand.groupby("category_en"):
    series = group.set_index("month")["units"]
    forecast_df, metrics = forecast_category(series, horizon=3)
    metrics["category_en"] = category
    forecast_metrics.append(metrics)
    if forecast_df is not None:
        forecast_df["category_en"] = category
        forecast_outputs.append(forecast_df)

forecast_metrics_df = pd.DataFrame(forecast_metrics)
display(forecast_metrics_df.sort_values("RF_WAPE_pct", na_position="last"))
forecast_table = pd.concat(forecast_outputs, ignore_index=True) if forecast_outputs else pd.DataFrame()
display(forecast_table.head(30))

### 7.3 Pricing & Margin What-If Model
A scenario calculator can quantify the arithmetic impact of price/discount/cost changes using observed line-level quantities and costs. It is not an elasticity model and does not estimate customer response or causal demand changes.

In [ ]:
pricing_base = items_clean.copy()
pricing_base["quantity"] = pd.to_numeric(pricing_base["quantity"], errors="coerce")
pricing_base["unit_price_EGP"] = pd.to_numeric(pricing_base["unit_price_EGP"], errors="coerce")
pricing_base["unit_cost_EGP"] = pd.to_numeric(pricing_base["unit_cost_EGP"], errors="coerce")
pricing_base = pricing_base.dropna(subset=["quantity", "unit_price_EGP", "unit_cost_EGP"])
pricing_base = pricing_base[(pricing_base["quantity"] > 0) & (pricing_base["unit_price_EGP"] >= 0)]
base_units = pricing_base["quantity"].sum()
base_revenue = (pricing_base["quantity"] * pricing_base["unit_price_EGP"]).sum()
base_cost = (pricing_base["quantity"] * pricing_base["unit_cost_EGP"]).sum()
scenarios = []
for price_change_pct, discount_change_pct, cost_change_pct in [
    (0, 0, 0), (5, 0, 0), (-5, 0, 0), (0, 2, 0), (0, 0, 5), (5, 1, -3)
]:
    scenario_price = pricing_base["unit_price_EGP"] * (1 + price_change_pct / 100)
    scenario_discount = np.maximum(0, scenario_price * discount_change_pct / 100)
    scenario_revenue = ((scenario_price - scenario_discount) * pricing_base["quantity"]).sum()
    scenario_cost = (pricing_base["unit_cost_EGP"] * (1 + cost_change_pct / 100) * pricing_base["quantity"]).sum()
    scenario_profit = scenario_revenue - scenario_cost
    scenarios.append({
        "price_change_pct": price_change_pct,
        "discount_pct_of_scenario_price": discount_change_pct,
        "unit_cost_change_pct": cost_change_pct,
        "scenario_revenue_EGP": scenario_revenue,
        "scenario_cost_EGP": scenario_cost,
        "scenario_gross_profit_EGP": scenario_profit,
        "scenario_margin_pct": scenario_profit / scenario_revenue * 100 if scenario_revenue else np.nan,
    })
pricing_scenarios = pd.DataFrame(scenarios)
display(pricing_scenarios)

### 7.4 Warehouse-to-Branch Allocation Plan
The stock table is aggregated at its documented business grain. Do not treat category/location monthly stock as SKU-level stock. The following is a diagnostic view of stock by location/category; any allocation recommendation must respect transfer direction and operational constraints.

In [ ]:
stock_view = stock.copy()
if "year_month" in stock_view:
    stock_view["year_month"] = pd.to_datetime(stock_view["year_month"], errors="coerce")
if "year_month" in stock_view and stock_view["year_month"].notna().any():
    latest_stock_month = stock_view["year_month"].max()
    latest_stock = stock_view[stock_view["year_month"].eq(latest_stock_month)].copy()
    print("Latest stock snapshot:", latest_stock_month)
else:
    latest_stock = stock_view.copy()
display(latest_stock.head())
stock_group_cols = [c for c in ["stock_location_type", "stock_location_id", "product_category"] if c in latest_stock]
stock_measure = next((c for c in ["closing_stock_units", "stock_units", "quantity", "on_hand_units"] if c in latest_stock), None)
if stock_group_cols and stock_measure:
    display(latest_stock.groupby(stock_group_cols, dropna=False)[stock_measure].sum().reset_index()
            .sort_values(stock_measure, ascending=False))
print("Transfer columns:", transfers.columns.tolist())
print("Transfer item columns:", transfer_items.columns.tolist())

### 7.5 High-Value Refund Statistical Review
Flag unusually large refunds for review using a robust IQR threshold. A statistical flag is not proof of fraud or error. Return records alone do not prove that returned goods were received back into inventory.

In [ ]:
refund_review = returns_clean.copy()
refund_review["refund_amount_EGP"] = pd.to_numeric(refund_review["refund_amount_EGP"], errors="coerce")
refund_values = refund_review["refund_amount_EGP"].dropna()
if len(refund_values):
    q1, q3 = refund_values.quantile([0.25, 0.75])
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr
    refund_review["high_value_flag"] = refund_review["refund_amount_EGP"] > upper_fence
    print("IQR upper fence (EGP):", round(upper_fence, 2))
    display(refund_review[refund_review["high_value_flag"]]
            .sort_values("refund_amount_EGP", ascending=False).head(25))
    display(refund_review.groupby("reason", dropna=False).agg(
        return_records=("return_id", "count"),
        total_refund_EGP=("refund_amount_EGP", "sum"),
        median_refund_EGP=("refund_amount_EGP", "median"),
        high_value_records=("high_value_flag", "sum")
    ).sort_values("total_refund_EGP", ascending=False))

### 7.6 Keyword Direct-Query Copilot — analytical contract
The current application uses a keyword/rule-based direct-query approach. Document supported entities, measures, filters and aggregations explicitly. It should not be described as an LLM or generative assistant unless the implementation is actually upgraded.

Suggested query domains to expose in the app:
- **Sales:** orders, gross/net sales, discounts, paid amount, units, monthly trend.
- **Branches:** branch sales, order volume, average order value and margin where valid.
- **Customers:** RFM segments and customer-level purchase history for identified customers.
- **Inventory:** latest monthly stock snapshot, stock location and product category.
- **Returns:** return units, refund amount, reasons and statistical high-value flags.
- **Forecasting:** category-level forecast and backtest metrics with the model/baseline clearly named.

Every answer should state its date range, grain, filters and metric definition. Unsupported questions should return a clear limitation rather than a guessed result.

## 8. Consolidated findings and data limitations
The following observations were identified during the audit. Re-run the cells above after any source refresh and update the findings if the data changes.

In [ ]:
audit_findings = pd.DataFrame([
    ["Customer identifiers", "102,961 sales orders (19.07%) have missing customer_id in the audited extract.",
     "Keep these orders in company totals; exclude only from customer-level RFM."],
    ["Data dictionary", "branches.csv and customers.csv have documented-vs-actual schema differences.",
     "Confirm whether the dictionary or source extract is authoritative before changing either."],
    ["Inventory grain", "monthly_stock is category/location/month grain, not SKU-level.",
     "Do not make SKU-level allocation claims from this table."],
    ["Returns", "A return/refund record does not confirm physical receipt into sellable stock.",
     "Reconcile with warehouse receiving/inspection records if available."],
    ["Procurement", "Purchase-order data does not by itself confirm quantities physically received.",
     "Do not report supplier fill rate or received quantity without receiving evidence."],
    ["Imports", "Import shipment data does not include SKU-level received units in the audited schema.",
     "Limit import analysis to fields actually recorded."],
    ["Delivery", "Ship/delivery timestamps can support elapsed transit-time analysis.",
     "Do not label shipments late without a promised date or defined SLA."],
    ["Marketing", "Campaign metrics are available, but an order-level campaign link is not established.",
     "Use descriptive campaign metrics; avoid causal attribution claims."],
    ["Partial period", "The latest observed sales month may be incomplete.",
     "Mark partial periods and avoid misleading full-month comparisons."],
], columns=["area", "observed_limitation", "analytical_treatment"])
display(audit_findings)

## 9. Application alignment and project conclusion

In [ ]:
application_alignment = pd.DataFrame([
    ["Warehouse-to-Branch Allocation Plan", "monthly_stock, warehouses, inventory_transfers, inventory_transfer_items",
     "Category/location stock diagnostics and transfer relationship checks",
     "Monthly stock is not SKU-level; confirm source/destination and receiving evidence."],
    ["Pricing & Margin What-If Model", "sales_order_items, products_catalog, sku_variants",
     "Scenario-based price, discount, cost and margin calculations",
     "Scenario arithmetic only; no price elasticity or demand response inferred."],
    ["Customer Win-Back / RFM", "sales_orders, customers",
     "Recency, frequency and monetary segmentation",
     "Missing customer IDs excluded from RFM but retained in total sales."],
    ["High-Value Refund Statistical Review", "sales_returns, sales_order_items",
     "Refund distribution, return reasons and outlier review",
     "Outliers are review flags, not fraud findings or inventory receipts."],
    ["AI Demand & Inventory Intelligence", "sales_order_items, products_catalog, sku_variants, monthly_stock",
     "Random Forest monthly demand forecast with six-month walk-forward and seasonal-naive benchmark",
     "Forecast requires sufficient history; category grain and data coverage must be stated."],
    ["Keyword Direct-Query Copilot", "Validated analytical tables and KPI definitions",
     "Rule/keyword-based query responses over supported metrics",
     "Not an LLM unless a generative model is actually integrated."],
], columns=["current_application", "main_data_sources", "notebook_coverage", "important_guardrail"])
display(application_alignment)

### Conclusion

The notebook documents the data landscape before modeling, validates key business relationships, reconciles order headers to sales lines, profiles payments/returns/delivery/procurement, and creates a controlled preparation layer. It then extends the analysis into the same six business use cases exposed by the current Cottonil application.

The analysis intentionally distinguishes **observed facts**, **derived metrics**, and **limitations**. Missing identifiers are not fabricated, unsupported operational measures are not inferred, and statistical/forecasting outputs are treated as decision support rather than certainty.

**Reproducibility:** keep the original CSV files unchanged, configure `COTTONIL_DATA_DIR` when running on another machine, execute cells from top to bottom, and record the source-data refresh date when publishing updated results.